Import libraries, set date range for analysis period, and initialize a `Sources` class object - it contains all the stock/index name and the corresponding tickers that we need. We also initialize `data_frames` to compile all DataFrames.

In [1]:
import requests
import json
import os
import sys
import time
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
from fredapi import Fred
from dotenv import load_dotenv

sys.path.append('..')
from modules.source import Sources, RAW_DATA_PATH, PROCESSED_DATA_PATH, daily_master_csv, monthly_master_csv
from modules.helpers import clean_api_response, parse_10y_entry
stock = Sources()

START = datetime(2015, 1, 1)
END = datetime(2026, 8, 25)
   
load_dotenv("../.env")
FRED_API_KEY = os.getenv('FRED_API_KEY')
if FRED_API_KEY is None: print('Enter your FRED API key in .env')
fred = Fred(FRED_API_KEY)

data_frames: dict[pd.DataFrame] = {}

Since these operations are identical across all tickers, from here on we only note material deviations from this structure.
1. **Fetch** - pull data from FRED
2. **Save** - write the raw response to `.csv`
3. **Process** - use `clean_api_response` to standardize data format, and adjust to add back publication lag (if any) 
4. **Add to `data_frames` object**

We convert the raw data to DataFrame before saving it purely for pipeline consistency. Saving it as a Series instead would produce an identical CSV.



In [2]:
series = fred.get_series_first_release(stock.palm_oil_global.ticker)

df = clean_api_response(series, stock.palm_oil_global)
df.index = pd.to_datetime(df.index)

df.to_csv(f"{RAW_DATA_PATH}/{stock.palm_oil_global}.csv")

df = df.loc[START:END]
data_frames[stock.palm_oil_global] = df

FFR is provided with a range, so we take the midpoint. The FRED series is based on effective date (which is 1 day later than announcement date), so we don't need to adjust for it - we just leave it out of timezone adjustment (also lags for 1 day).

In [3]:
series_upper = fred.get_series('DFEDTARU')
series_lower = fred.get_series('DFEDTARL')

df = pd.concat([series_upper, series_lower], axis=1, join='inner')
df.columns = ['Upper', 'Lower']
df['Midpoint'] = (df['Upper'] + df['Lower']) / 2
df.index.name = 'date'
df.to_csv(f"{RAW_DATA_PATH}/{stock.FFR_midpoint}.csv")

df = df[['Midpoint']].rename(columns={'Midpoint': stock.FFR_midpoint})
df = df.loc[START:END]
data_frames[stock.FFR_midpoint] = df

The EFFR data is already adjusted for 1 day publication lag, so we shift it back to the publication date.

In [4]:
series = fred.get_series(stock.EFFR.ticker)
df.to_csv(f"{RAW_DATA_PATH}/{stock.EFFR}.csv")

df = clean_api_response(series, stock.EFFR)

lag = timedelta(days=1)
df = df.loc[START - lag : END - lag]
df = df.shift(1, lag)

data_frames[stock.EFFR] = df

We pull the data from yfinance, and use the daily closing price as our price data.

In [5]:
yahoo_stocks = [stock.UST_10Y, stock.VIX, stock.USDMYR, stock.DXY, stock.brent_oil, stock.KLCI]

for name in yahoo_stocks:
   df = yf.download(name.ticker, START, END, progress=False)
   df.to_csv(f'{RAW_DATA_PATH}/{name}.csv')

   try:
      df = clean_api_response(df, name)
   except ValueError as e:
      print(f"  FAILED - {name}: {e}")
      continue

   print(f"{name} ({name.ticker}) - saved {len(df)} rows")
   df = df[['Close']].rename(columns={'Close': name})
   data_frames[name] = df

UST_10Y (^TNX) - saved 2926 rows
VIX (^VIX) - saved 2928 rows
USDMYR (MYR=X) - saved 3031 rows
DXY (DX-Y.NYB) - saved 2928 rows
Brent_Oil (BZ=F) - saved 2928 rows
KLCI (^KLSE) - saved 2850 rows


For CPI inflation by DOSM, we narrow it down to overall division only and YoY inflation as our primary series. And undo the ~2 month of publication lag.

One thing to note: core CPI data is only available from 2018 onwards, so YoY (needing a 12-month lookback) is only available from 2019 onwards - this affects CPI-related analysis specifically, not the full project's date range.

In [6]:
df_cpi = pd.read_parquet('https://storage.dosm.gov.my/cpi/cpi_2d_core_inflation.parquet')
df_cpi.to_csv(f'{RAW_DATA_PATH}/CPI.csv')

df_cpi = (
   df_cpi
   .assign(date=pd.to_datetime(df_cpi['date']))
   .set_index('date')
   .query("division == 'overall'")
   .drop(columns=['division', 'inflation_mom'])
   .shift(2, freq='ME')
   .loc[START:END]
   .dropna(how='any')
   .rename(columns={'inflation_yoy': stock.cpi_inflation_yoy})
)

data_frames[stock.cpi_inflation_yoy] = df_cpi

As with core CPI, for OPR, we set `date` as the index, then sort it before saving. The year-by-year fetch doesn't guarantee chronological order.

Post-save, we drop `year` (redundant once `date` is the index) and `change_in_opr` (we'll reconstruct this ourselves downstream), and rename `new_opr_level` to `OPR`.

In [7]:
headers = {'Accept': 'application/vnd.BNM.API.v1+json'}

records = []
for year in range(START.year, END.year + 1):
   resp = requests.get(f'https://api.bnm.gov.my/public/opr/year/{year}', headers=headers)
   records.extend(resp.json()['data'])

df_opr = pd.DataFrame(records)
df_opr['date'] = pd.to_datetime(df_opr['date'])
df_opr = df_opr.set_index('date')
df_opr = df_opr.sort_index()

df_opr = df_opr[~df_opr.index.duplicated(keep='last')]
df_opr.to_csv(f'{RAW_DATA_PATH}/{stock.OPR}.csv')

df_opr = (
   df_opr.loc[START:END]
   .drop(columns=['year', 'change_in_opr'])
   .rename(columns={'new_opr_level': stock.OPR})
)

data_frames[stock.OPR] = df_opr

In [ ]:
BASE_URL = "https://api.bnm.gov.my/public/gov-sec-yield"
HEADERS = {"Accept": "application/vnd.BNM.API.v1+json"}

business_days = pd.bdate_range(START, END)  # Mon-Fri only; holidays still queried but expected empty

records = []
no_data_days = []   # likely public holidays
failed_days = []    # genuine API/network failures - worth re-running individually

for year, year_group in business_days.to_series().groupby(business_days.year):
   year_records = 0
   for d in year_group:
      date_str = d.strftime("%Y-%m-%d")
      resp = requests.get(BASE_URL, headers=HEADERS, params={"date": date_str}, timeout=30)

      if resp.status_code != 200:
         try:
            err = resp.json()
            print(f"[{date_str}] Failed: {err.get('code', resp.status_code)} - {err.get('message', 'Unknown error')}")
         except ValueError:
            print(f"[{date_str}] Failed: HTTP {resp.status_code} - {resp.text[:200]}")
         failed_days.append(date_str)
         time.sleep(0.3)
         continue

      payload = resp.json()
      entry = parse_10y_entry(payload, date_str)
      if entry is not None:
         records.append(entry)
         year_records += 1
      else:
         no_data_days.append(date_str)

      time.sleep(0.3)

   print(f"[{year}] {year_records} of {len(year_group)} business days have a 10Y entry")

mgs_10y_daily = pd.DataFrame(records).sort_values("date").reset_index(drop=True)
mgs_10y_daily.to_csv(f"{RAW_DATA_PATH}/{stock.MGS_10Y}.csv", index=False)

print(f"\nTotal: {len(mgs_10y_daily)} daily 10Y observations")
print(f"Business days with no 10Y entry (likely public holidays): {len(no_data_days)}")
print(f"Failed API calls: {len(failed_days)}")
if failed_days:
   print("Failed dates (re-run these individually):", failed_days)

df = mgs_10y_daily.drop(columns=['tot_vol', 'maturity_month', 'maturity_year', 'daily_change']).set_index('date')
df.index = pd.to_datetime(df.index)
data_frames[stock.MGS_10Y] = df

[2015] 241 of 261 business days have a 10Y entry
[2016] 238 of 261 business days have a 10Y entry
[2017] 217 of 260 business days have a 10Y entry
[2018] 227 of 261 business days have a 10Y entry
[2019] 232 of 261 business days have a 10Y entry
[2020] 247 of 262 business days have a 10Y entry
[2021] 238 of 261 business days have a 10Y entry
[2022] 235 of 260 business days have a 10Y entry
[2023] 241 of 260 business days have a 10Y entry
[2024] 247 of 262 business days have a 10Y entry
[2025] 233 of 261 business days have a 10Y entry
[2026] 155 of 169 business days have a 10Y entry

Total: 2751 daily 10Y observations
Business days with no 10Y entry (likely public holidays): 288
Failed API calls: 0


,date,yield_close,daily_change,tot_vol,maturity_month,maturity_year
0,2015-01-02,4.13,0.05,30.00,7,2024
1,2015-01-05,4.17,0.04,234.14,7,2024
2,2015-01-06,4.20,0.03,45.20,7,2024
3,2015-01-07,4.17,-0.03,20.93,7,2024
4,2015-01-08,4.19,0.02,172.45,7,2024


For each sector, we aggregate 2-3 large, widely-held constituents, save to CSV, then reconstruct its equal-weighted index. For plantation, we trim data from before 2017-11-30 (SD Gurthie Berhad hasn't listed yet) or else it would produce price-level break.

In [15]:
sector_constituents = {
   stock.financials: ['1155.KL', '1023.KL', '1295.KL'],   # Maybank, CIMB, Public Bank
   stock.plantation: ['5285.KL', '1961.KL', '2445.KL'],   # Sime Darby Plantation, IOI Corp, KLK
   stock.reits:      ['5227.KL', '5176.KL', '5235SS.KL'], # IGB REIT, Sunway REIT, KLCCP Stapled
   stock.technology: ['0166.KL', '0097.KL', '0128.KL'],   # Inari Amertron, ViTrox, Frontken
   stock.energy:     ['6033.KL', '5183.KL', '5681.KL'],   # PetGas, PetChem, Petronas Dagangan
   stock.industrial_products: ['8869.KL', '7113.KL'],     # Press Metal, Top Glove
}

for sector, tickers in sector_constituents.items():
   print(f"\n=== {sector.upper()} ===")
   closing_price_series: dict[pd.Series] = {}
   
   for t in tickers:
      df = yf.download(t, START, END, progress=False)
      try:
         df = clean_api_response(df, t)
      except ValueError as e:
         print(f"  FAILED - {t}: {e}")
         continue

      closing_price_series[t] = df['Close'] # stored as series
      print(f"  OK - {t} ({len(df)} rows)")

   sector_df = pd.DataFrame(closing_price_series)
   sector_df.to_csv(f'{RAW_DATA_PATH}/sectors/{sector}.csv')
   sector_df.index.name = 'date'

   if sector is stock.plantation:
      cutoff = datetime(2017, 11, 30) # 5285.KL (SD Guthrie Berhad) date of being listed
      sector_df = sector_df.loc[cutoff:]

   sector_df[sector] = sector_df.ffill().pct_change().mean(axis=1)
   sector_df[sector] = 100 * (1 + sector_df[sector].fillna(0)).cumprod()
   data_frames[sector] = sector_df[sector]


=== FINANCIALS ===
  OK - 1155.KL (2868 rows)
  OK - 1023.KL (2869 rows)
  OK - 1295.KL (2869 rows)

=== PLANTATION ===
  OK - 5285.KL (2142 rows)
  OK - 1961.KL (2868 rows)
  OK - 2445.KL (2868 rows)

=== REITS ===
  OK - 5227.KL (2867 rows)
  OK - 5176.KL (2868 rows)
  OK - 5235SS.KL (2868 rows)

=== TECHNOLOGY ===
  OK - 0166.KL (2868 rows)
  OK - 0097.KL (2869 rows)
  OK - 0128.KL (2869 rows)

=== ENERGY ===
  OK - 6033.KL (2868 rows)
  OK - 5183.KL (2868 rows)
  OK - 5681.KL (2869 rows)

=== INDUSTRIAL_PRODUCTS ===
  OK - 8869.KL (2868 rows)
  OK - 7113.KL (2868 rows)


Now we adjust for timezone difference (look-ahead bias).

In [29]:
us_market_ticker = [stock.EFFR, stock.UST_10Y, stock.USDMYR, stock.DXY, stock.VIX, stock.brent_oil, stock.palm_oil_global]

for i, df in data_frames.items():
   df = df.ffill()
   lag = timedelta(days=1)
   if i in us_market_ticker:
      df = df.shift(1, lag).loc[START:]
   data_frames[i] = df.loc[START:(END + lag)]

Combine to master DataFrame and save to CSV.

In [30]:
daily_master: pd.DataFrame = pd.concat(data_frames.values(), axis=1, sort=True)
daily_master.to_csv(f"{PROCESSED_DATA_PATH}/{daily_master_csv}")

In [31]:
monthly_master = daily_master.ffill().resample("ME").last()
monthly_master.to_csv(f"{PROCESSED_DATA_PATH}/{monthly_master_csv}")